# LO8 캡스톤. Agent 장기 메모리와 회상 전략

하루 동안 만든 부품(모델, 도구, 그래프, 단기 메모리)에 장기 메모리를 더해 "기억하는 Agent"를 완성하는 캡스톤 실습입니다. 단계별로 실행하고 각 체크포인트를 확인하십시오.

## 학습 목표

- 장기 메모리 Store의 put, get, search 동작과 시맨틱 인덱스의 역할을 설명할 수 있다
- 회상 전략(최근성, 관련성, 중요도)과 저장 위치(Tool-call, In-graph)를 비교해 선택할 수 있다
- 단기 메모리(checkpointer)와 장기 메모리(Store)를 결합한 종합 Agent를 구성하고 실행할 수 있다

주 모델은 `openai:gpt-5.4-mini`, 임베딩은 `openai:text-embedding-3-small`(1536차원)을 사용합니다.

## 단계 0. 환경 준비 (핀 고정 설치)

검증된 버전으로 핀을 고정해 설치합니다. 강의 직전 모델명과 가격을 다시 확인하십시오.

In [ ]:
# 검증된 버전으로 핀 고정
!pip install -U "langchain==1.3.4" "langgraph==1.2.4" langchain-openai langchain-google-genai

In [ ]:
import os

# Colab 좌측 열쇠 메뉴에 한 번 등록하면 이후 모든 LO에서 재사용됩니다
# 임베딩과 모델 모두 OpenAI 키를 사용합니다
from google.colab import userdata
os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

# 로컬 또는 IDE에서 실행할 때는 위 두 줄 대신 아래처럼 환경변수를 직접 사용하십시오
# os.environ["OPENAI_API_KEY"] = "sk-..."  # 또는 .env 파일을 python-dotenv 로 로드

> 체크포인트 0. 설치 후 오류 없이 import가 되면 통과입니다.

## 단계 1. 시맨틱 Store에 저장하고 회상하기

시맨틱 인덱스를 켠 Store를 만들면 저장하는 순간 텍스트가 임베딩 벡터로 변환됩니다. 이후 키를 몰라도 자연어로 의미가 가까운 기억을 회상할 수 있습니다.

In [ ]:
from langgraph.store.memory import InMemoryStore
from langchain.embeddings import init_embeddings  # v1 경로

# 시맨틱 인덱스를 켠 Store 생성. 저장 시 텍스트를 1536차원 벡터로 변환합니다
store = InMemoryStore(
    index={
        "dims": 1536,  # text-embedding-3-small 차원
        "embed": init_embeddings("openai:text-embedding-3-small"),  # 임베딩 모델 (dims와 한 쌍)
        "fields": ["text"],  # 임베딩할 필드 지정 (value의 text만 벡터화, 메타데이터 제외)
    }
)

ns = ("user-123", "memories")  # (사용자, 주제)로 기억 공간 분리. 첫 칸이 사용자 ID
store.put(ns, "fact-1", {"text": "앤디는 파이썬을 좋아한다"})  # 저장
store.put(ns, "fact-2", {"text": "앤디는 매운 음식을 못 먹는다"})

print(store.get(ns, "fact-1").value)  # 정확한 키 조회: {'text': '앤디는 파이썬을 좋아한다'}

# 의미 기반 회상. 키를 몰라도 자연어로 검색하면 가까운 순서대로 돌려줍니다
for item in store.search(ns, query="좋아하는 프로그래밍 언어"):
    # item.score 에 유사도가 담깁니다 (높을수록 가까움)
    print(round(item.score, 3), item.value["text"])
# 예상 출력 (점수는 모델, 버전에 따라 다를 수 있음):
#   0.41 앤디는 파이썬을 좋아한다       질의와 의미가 가까워 먼저
#   0.13 앤디는 매운 음식을 못 먹는다    의미가 멀어 뒤

변형 포인트. `query`를 `"좋아하는 음식"`으로 바꾸면 두 기억의 순서가 뒤집힙니다. 같은 저장소라도 질문이 무엇이냐에 따라 회상 순서가 달라진다는 점을 직접 확인하십시오. 키워드가 아니라 의미로 정렬한다는 증거입니다.

> 체크포인트 1. "좋아하는 프로그래밍 언어"로 검색했을 때 음식 기억보다 파이썬 기억이 먼저(점수가 더 높게) 회상되면 통과입니다(의미 기반 정렬 확인).

> 주의. `dims`(1536)는 `embed`에 지정한 임베딩 모델과 한 쌍입니다. text-embedding-3-small은 1536차원이지만, 다른 모델로 바꾸면 그 모델의 출력 차원에 맞춰 `dims`도 함께 고쳐야 합니다. 둘이 어긋나면 저장, 검색 시 차원 오류가 납니다.

## 단계 1.5. 같은 키로 사실 갱신하기 (망각과 갱신)

바뀌는 사실은 새 키로 또 저장하지 않고, 사실 종류마다 정한 고정 키를 덮어씁니다. 더 이상 필요 없는 기억은 delete로 망각합니다.

In [ ]:
# 거주지는 바뀌는 정보이므로, 새 키가 아니라 고정 키 "residence"에 덮어씁니다
store.put(ns, "residence", {"text": "앤디는 서울에 산다"})
print(store.get(ns, "residence").value)  # {'text': '앤디는 서울에 산다'}

# 이사 후. 새 fact를 추가하지 않고 같은 키를 덮어씁니다 (옛 기억이 교체됨)
store.put(ns, "residence", {"text": "앤디는 부산으로 이사했다"})
print(store.get(ns, "residence").value)  # {'text': '앤디는 부산으로 이사했다'} (서울 기억은 사라짐)

# 더 이상 필요 없는 기억은 삭제로 망각합니다
store.delete(ns, "fact-2")  # '매운 음식' 기억 제거
print(store.get(ns, "fact-2"))  # None (지워짐)

변형 포인트. 갱신을 `store.put(ns, "residence-2", ...)`처럼 새 키로 저장해 보면, 이후 `search("어디 사는지")`에 서울과 부산이 함께 회상되어 모순이 생깁니다. 바뀌는 사실에는 고정 키 덮어쓰기가 왜 안전한지 비교로 확인하십시오.

> 체크포인트 1.5. 같은 키 `"residence"`를 두 번 put했을 때 마지막 값만 남고, delete한 키는 get이 None을 돌려주면 통과입니다(갱신과 망각 동작 확인).

## 단계 2. 단기 + 장기 메모리 결합 종합 Agent

In-graph 방식으로 노드가 직접 Store를 호출해 회상합니다. compile 시 넘긴 store가 노드 인자로 자동 주입되며, 단기 메모리(checkpointer)와 장기 메모리(store)를 한 Agent에 함께 답니다.

In [ ]:
from typing import Annotated
from typing_extensions import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.checkpoint.memory import InMemorySaver  # 단기 메모리
from langgraph.store.base import BaseStore
from langchain.chat_models import init_chat_model
from langchain.messages import SystemMessage  # v1 경로

model = init_chat_model("openai:gpt-5.4-mini")  # 강의 직전 최신 모델, 가격 재확인


class State(TypedDict):
    messages: Annotated[list, add_messages]  # 단기: 메시지 누적


# In-graph 방식. 노드가 직접 Store를 호출합니다. compile 시 store가 인자로 자동 주입됩니다(*, store)
def agent_node(state: State, *, store: BaseStore):
    last = state["messages"][-1].content
    # 1) 회상. 마지막 사용자 발화와 의미가 가까운 장기 기억을 상위 3개만 가져옵니다
    recalled = store.search(("user-123", "memories"), query=last, limit=3)
    memory_text = "\n".join(f"- {r.value['text']}" for r in recalled) or "(없음)"
    # 회상한 기억을 시스템 프롬프트에 넣되, 부정과 시점은 LLM이 직접 해석하도록 맡깁니다
    sys = SystemMessage(
        "너는 사용자를 기억하는 비서다. 아래는 회상한 사실이며, "
        "질문과 어긋나면 무시하라.\n" + memory_text
    )
    # 2) 응답. 회상한 기억을 시스템 프롬프트에 끼워 답변합니다
    reply = model.invoke([sys] + state["messages"])
    return {"messages": [reply]}


# 변형 포인트. limit를 1로 줄이면 회상이 줄어 답이 빈약해지고, 키우면 노이즈가 늘어
#            엉뚱한 기억까지 끼어듭니다. 회상 개수는 품질과 토큰의 트레이드오프입니다.

b = StateGraph(State)
b.add_node("agent", agent_node)
b.add_edge(START, "agent")
b.add_edge("agent", END)

# 단기(checkpointer) + 장기(store)를 함께 장착합니다
checkpointer = InMemorySaver()
agent = b.compile(checkpointer=checkpointer, store=store)

> 체크포인트 2. compile에 checkpointer와 store를 모두 넘겼고, 노드가 `*, store` 인자로 Store를 받는 구조면 통과입니다.

## 단계 3. 새 세션에서도 기억하는지 검증

같은 thread 안에서는 단기 메모리가 직전 발화를 기억하고, 새 thread에서는 단기 맥락이 비어 있습니다. 반면 Store에 저장한 장기 기억은 thread를 넘어 회상됩니다.

In [ ]:
# thread A. 단기 메모리로 대화 맥락을 누적합니다
config_a = {"configurable": {"thread_id": "session-A"}}

# (1) 사용자가 방금 한 발화는 같은 thread 안에서만 기억되는 단기 정보입니다
agent.invoke({"messages": [{"role": "user", "content": "오늘 점심에 김치찌개를 먹었어"}]}, config_a)
# (2) 바로 이어지는 질문. 단기 메모리(checkpointer)가 같은 thread의 직전 발화를 기억합니다
res_a = agent.invoke({"messages": [{"role": "user", "content": "내가 점심에 뭐 먹었다고 했지?"}]}, config_a)
print(res_a["messages"][-1].content)  # 같은 thread의 단기 맥락에서 '김치찌개'를 회상
# 예상 출력: "점심에 김치찌개를 드셨다고 하셨습니다." (직전 발화가 checkpointer에 남아 있음)

# 새 기억을 장기 메모리에 추가 (In-graph 저장은 노드에 넣을 수도 있으나, 여기선 직접 저장으로 확인)
store.put(("user-123", "memories"), "fact-3", {"text": "앤디는 주말마다 등산을 간다"})

# thread B. 완전히 새 세션 (단기 메모리는 비었지만 장기 메모리는 공유)
config_b = {"configurable": {"thread_id": "session-B"}}

# (3) 단기 정보 확인. thread A의 점심 발화는 thread B에 없으므로 회상하지 못합니다
res_b1 = agent.invoke({"messages": [{"role": "user", "content": "내가 점심에 뭐 먹었다고 했지?"}]}, config_b)
print(res_b1["messages"][-1].content)  # thread가 달라 단기 맥락이 없어 점심을 알지 못함
# 예상 출력: "어떤 점심을 드셨는지 알지 못합니다." (새 thread라 단기 맥락이 비어 있음)

# (4) 장기 정보 확인. 등산 기억은 Store에 영속하므로 새 세션에서도 회상합니다
res_b2 = agent.invoke({"messages": [{"role": "user", "content": "주말에 내가 보통 뭐 한다고 했지?"}]}, config_b)
print(res_b2["messages"][-1].content)  # 장기 메모리에서 '등산'을 회상해 답변
# 예상 출력: "주말마다 등산을 가신다고 알고 있습니다." (Store에 영속한 장기 기억)

변형 포인트. thread B의 namespace를 `("user-999", "memories")`로 바꿔 회상해 보면, 등산 기억은 user-123 공간에만 있으므로 회상되지 않습니다. namespace로 사용자 기억이 어떻게 격리되는지 직접 확인하십시오.

> 체크포인트 3. 같은 thread A에서는 단기 메모리로 '김치찌개'를 회상하지만, 새 thread B에서는 그 점심 기억을 알지 못합니다. 반면 장기 메모리에 저장한 등산 기억은 thread B에서도 회상합니다. 단기와 장기의 차이가 드러나면 캡스톤 완성입니다.

> 주의. InMemorySaver와 InMemoryStore는 모두 프로세스 메모리에 저장되어 런타임 재시작 시 사라집니다. 데모용이며, 운영에서는 PostgresSaver, PostgresStore 등 영속 백엔드로 교체합니다.

오늘 배운 LLM(추론), 도구(행동), 그래프(제어 흐름), 단기와 장기 메모리(상태)가 모두 한 Agent에 모였습니다. 이것으로 Agent 4요소가 완성됩니다.